# AI-Driven Portfolio Optimizer Walkthrough

This notebook demonstrates an end-to-end workflow for an **AI-driven portfolio optimization** project:

1. Download historical stock prices with `yfinance`
2. Engineer technical indicators
3. Cluster stocks into risk groups with **K-Means**
4. Predict next-day returns with **Random Forest**
5. Optimize allocations with **PyPortfolioOpt**
6. Backtest strategies over a 5-year period
7. Visualize results with **Plotly**


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import DEFAULT_TICKERS, RISK_LEVEL_MAP, TECHNICAL_FEATURES
from src.data_loader import download_price_data, compute_returns
from src.features import build_technical_features
from src.ml_models import cluster_stocks, prepare_ml_dataset, train_random_forest_models, predict_next_day_scores
from src.optimizer import optimize_portfolio
from src.backtest import run_backtest, summarize_performance
from src.visuals import plot_price_history, plot_cluster_map, plot_cumulative_returns, plot_feature_importance, plot_weights

print('Project modules imported successfully.')

## 1. Define the stock universe and download historical data

In [ ]:
tickers = DEFAULT_TICKERS
start_date = '2021-01-01'
end_date = None

prices = download_price_data(tickers, start=start_date, end=end_date)
returns = compute_returns(prices)

prices.tail()

## 2. Explore historical price performance

In [ ]:
plot_price_history(prices)

## 3. Cluster stocks by risk using K-Means

We summarize each stock using annualized return, annualized volatility, Sharpe ratio, and downside volatility, then assign a risk label based on cluster volatility.

In [ ]:
clusters = cluster_stocks(returns)
clusters

In [ ]:
plot_cluster_map(clusters)

## 4. Build technical indicators for machine learning

In [ ]:
feature_panel = {ticker: build_technical_features(prices[ticker]) for ticker in prices.columns}
dataset = prepare_ml_dataset(feature_panel)

dataset.head()

## 5. Train Random Forest models

We train:
- a **regressor** for next-day return
- a **classifier** for next-day up/down direction


In [ ]:
regressor, classifier = train_random_forest_models(dataset)

latest_rows = []
for ticker, df in feature_panel.items():
    row = df.tail(1).copy()
    row['ticker'] = ticker
    latest_rows.append(row)
latest_feature_rows = __import__('pandas').concat(latest_rows)

predictions = predict_next_day_scores(latest_feature_rows, regressor, classifier)
predictions

In [ ]:
plot_feature_importance(regressor, TECHNICAL_FEATURES)

## 6. Optimize a portfolio with PyPortfolioOpt

We create two optimized portfolios:
- **AI-Optimized**: blends historical expected returns with ML-predicted alpha
- **Classical Mean-Variance**: uses historical expected returns only


In [ ]:
risk_target = RISK_LEVEL_MAP['Balanced']
ai_alpha = predictions.set_index('ticker')['predicted_return'] * 252
selected = predictions.query('prob_up >= 0.5')['ticker'].tolist()
if len(selected) < 3:
    selected = predictions.head(6)['ticker'].tolist()

ai_weights, ai_perf = optimize_portfolio(prices[selected], risk_target=risk_target, ai_alpha=ai_alpha)
mv_weights, mv_perf = optimize_portfolio(prices, risk_target=risk_target, ai_alpha=None)

ai_perf, mv_perf

In [ ]:
plot_weights(ai_weights, title='AI-Optimized Portfolio Weights')

## 7. Backtest strategies over time

The backtest compares:
- AI-Optimized
- MeanVariance
- EqualWeight


In [ ]:
backtest_returns, weights_history = run_backtest(prices, risk_target=risk_target)
performance_summary = summarize_performance(backtest_returns)

performance_summary

In [ ]:
plot_cumulative_returns(backtest_returns)

## 8. Interpretation

You can use this project in a portfolio to demonstrate:

- financial data collection and preprocessing
- feature engineering for time series
- unsupervised learning with K-Means
- supervised learning with Random Forest
- portfolio optimization with PyPortfolioOpt
- backtesting and strategy evaluation
- dashboard deployment with Streamlit


## Suggested Resume / Portfolio Description

**AI-Driven Portfolio Optimizer** — Built an end-to-end portfolio analytics app using `yfinance`, `scikit-learn`, `PyPortfolioOpt`, `Plotly`, and `Streamlit`. Engineered technical indicators, clustered stocks by risk using K-Means, predicted next-day returns with Random Forest, optimized allocations with mean-variance optimization, and backtested AI-assisted strategies over a 5-year horizon.